# 02 — Join Validation & Datamart Build

Notebook này kiểm tra quan hệ giữa các bảng và xây các bảng trung gian cho EDA/modeling.

Output chính:

- `data/interim/1_fact_order_item_enriched.csv`
- `data/interim/2_fact_order_enriched.csv`
- `data/interim/orders_payments_merged.csv`
- `data/interim/items_orders_merged.csv`
- `data/interim/items_products_merged.csv`
- `data/interim/items_returns_merged.csv`
- `data/interim/items_reviews_merged.csv`
- `data/interim/orders_customers_merged.csv`
- `data/interim/orders_geography_merged.csv`
- `data/interim/orders_shipments_final.csv`
- `data/marts/3_customer_360.csv`
- `data/marts/4_product_360.csv`
- `data/marts/5_daily_business_panel.csv`
- `data/marts/6_product_month_panel.csv`

Lưu ý: đây là notebook kiểm soát grain/key trước khi EDA. Một vài file item-level có thể có duplicate ở grain `(order_id, product_id)` nếu một đơn có nhiều dòng cùng sản phẩm hoặc do join returns/reviews 1:n; khi đó notebook sẽ ghi rõ là `CHECK` thay vì tự động xem là lỗi nghiêm trọng.

In [ ]:
# ============================================================
# Shared setup
# ============================================================
from pathlib import Path
import os
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# The notebooks are designed to run from repo/notebooks.
# They also work in Colab if you point PROJECT_ROOT or DATA_ROOT manually.
try:
    NOTEBOOK_DIR = Path.cwd().resolve()
except Exception:
    NOTEBOOK_DIR = Path(".").resolve()

PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MARTS_DIR = PROJECT_ROOT / "data" / "marts"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

for d in [RAW_DIR, INTERIM_DIR, MARTS_DIR, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [RAW_DIR, INTERIM_DIR, MARTS_DIR, PROCESSED_DIR, PROJECT_ROOT, Path("/mnt/data")]

def find_csv(filename: str) -> Path:
    """Find a CSV across standard repo folders and /mnt/data fallback."""
    for d in SEARCH_DIRS:
        p = d / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Cannot find {filename}. Checked: {[str(d) for d in SEARCH_DIRS]}")

def read_csv(filename: str, parse_dates=None, **kwargs) -> pd.DataFrame:
    path = find_csv(filename)
    return pd.read_csv(path, parse_dates=parse_dates, low_memory=False, **kwargs)

def save_table(df: pd.DataFrame, filename: str, index: bool = False) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=index)
    print(f"Saved table: {path}")
    return path

def save_fig(name: str, dpi: int = 160):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path

def money_fmt(x):
    if pd.isna(x):
        return "NA"
    if abs(x) >= 1e9:
        return f"{x/1e9:.2f}B"
    if abs(x) >= 1e6:
        return f"{x/1e6:.2f}M"
    if abs(x) >= 1e3:
        return f"{x/1e3:.2f}K"
    return f"{x:.2f}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)
print("MARTS_DIR:", MARTS_DIR)

PROJECT_ROOT: C:\datathon
RAW_DIR: C:\datathon\data\raw
INTERIM_DIR: C:\datathon\data\interim
MARTS_DIR: C:\datathon\data\marts


## 1. Load raw tables

In [ ]:
products = read_csv("products.csv")
customers = read_csv("customers.csv", parse_dates=["signup_date"])
promotions = read_csv("promotions.csv", parse_dates=["start_date", "end_date"])
geography = read_csv("geography.csv")
orders = read_csv("orders.csv", parse_dates=["order_date"])
items = read_csv("order_items.csv")
payments = read_csv("payments.csv")
shipments = read_csv("shipments.csv", parse_dates=["ship_date", "delivery_date"])
returns = read_csv("returns.csv", parse_dates=["return_date"])
reviews = read_csv("reviews.csv", parse_dates=["review_date"])
sales = read_csv("sales.csv", parse_dates=["Date"])
inventory = read_csv("inventory.csv", parse_dates=["snapshot_date"])
traffic = read_csv("web_traffic.csv", parse_dates=["date"])

raw_shapes = pd.DataFrame({
    "table": ["products", "customers", "promotions", "geography", "orders", "order_items", "payments", "shipments", "returns", "reviews", "sales", "inventory", "web_traffic"],
    "rows": [len(products), len(customers), len(promotions), len(geography), len(orders), len(items), len(payments), len(shipments), len(returns), len(reviews), len(sales), len(inventory), len(traffic)],
    "cols": [products.shape[1], customers.shape[1], promotions.shape[1], geography.shape[1], orders.shape[1], items.shape[1], payments.shape[1], shipments.shape[1], returns.shape[1], reviews.shape[1], sales.shape[1], inventory.shape[1], traffic.shape[1]],
})
raw_shapes

,table,rows,cols
0,products,2412,8
1,customers,121930,7
2,promotions,50,10
3,geography,39948,4
4,orders,646945,8
5,order_items,714669,7
6,payments,646945,4
7,shipments,566067,4
8,returns,39939,7
9,reviews,113551,7


## 2. Cardinality checks before joining

In [ ]:
cardinality_checks = []

def key_profile(df, table, key_cols, expected_unique=False):
    dup = int(df.duplicated(key_cols).sum())
    nulls = int(df[key_cols].isna().any(axis=1).sum())
    status = "PASS" if (not expected_unique or dup == 0) and nulls == 0 else "CHECK"
    cardinality_checks.append({
        "table": table,
        "key_cols": ", ".join(key_cols),
        "rows": len(df),
        "distinct_keys": df[key_cols].drop_duplicates().shape[0],
        "duplicate_key_rows": dup,
        "null_key_rows": nulls,
        "expected_unique": expected_unique,
        "status": status,
    })

key_profile(products, "products", ["product_id"], True)
key_profile(customers, "customers", ["customer_id"], True)
key_profile(promotions, "promotions", ["promo_id"], True)
key_profile(geography, "geography", ["zip"], True)
key_profile(orders, "orders", ["order_id"], True)
key_profile(items, "order_items", ["order_id", "product_id"], False)
key_profile(payments, "payments", ["order_id"], True)
key_profile(shipments, "shipments", ["order_id"], True)
key_profile(returns, "returns", ["return_id"], True)
key_profile(reviews, "reviews", ["review_id"], True)
key_profile(sales, "sales", ["Date"], True)
key_profile(inventory, "inventory", ["snapshot_date", "product_id"], True)
key_profile(traffic, "web_traffic", ["date", "traffic_source"], True)

cardinality = pd.DataFrame(cardinality_checks)
save_table(cardinality, "02_cardinality_checks.csv")
cardinality

Saved table: C:\datathon\reports\tables\02_cardinality_checks.csv


,table,key_cols,rows,distinct_keys,duplicate_key_rows,null_key_rows,expected_unique,status
0,products,product_id,2412,2412,0,0,True,PASS
1,customers,customer_id,121930,121930,0,0,True,PASS
2,promotions,promo_id,50,50,0,0,True,PASS
3,geography,zip,39948,39948,0,0,True,PASS
4,orders,order_id,646945,646945,0,0,True,PASS
5,order_items,"order_id, product_id",714669,714653,16,0,False,PASS
6,payments,order_id,646945,646945,0,0,True,PASS
7,shipments,order_id,566067,566067,0,0,True,PASS
8,returns,return_id,39939,39939,0,0,True,PASS
9,reviews,review_id,113551,113551,0,0,True,PASS


## 3. Helper functions for safe merges

In [ ]:
join_logs = []

def log_join(name, left_rows, right_rows, out_rows, left_key, right_key, expected=None, note=""):
    growth = out_rows / max(left_rows, 1)
    join_logs.append({
        "join_name": name,
        "left_rows": left_rows,
        "right_rows": right_rows,
        "output_rows": out_rows,
        "growth_vs_left": growth,
        "left_key": left_key,
        "right_key": right_key,
        "expected": expected or "",
        "status": "PASS" if growth <= 1.05 or expected == "1:n allowed" else "CHECK",
        "note": note,
    })

def save_interim(df, filename):
    path = INTERIM_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved {filename}: {df.shape}")
    return path

def save_mart(df, filename):
    path = MARTS_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved {filename}: {df.shape}")
    return path

## 4. Build base merged tables

In [ ]:
# 4.1 orders ↔ payments: expected 1:1
orders_payments = orders.merge(payments, on="order_id", how="left", suffixes=("", "_payment"), indicator=True)
log_join("orders_payments", len(orders), len(payments), len(orders_payments), "order_id", "order_id", "1:1")
orders_payments["payment_join_status"] = orders_payments.pop("_merge").astype(str)
save_interim(orders_payments, "orders_payments_merged.csv")

# 4.2 items ↔ orders: many item rows to one order
items_orders = items.merge(orders, on="order_id", how="left", validate="m:1", indicator=True)
log_join("items_orders", len(items), len(orders), len(items_orders), "order_id", "order_id", "m:1")
items_orders["order_join_status"] = items_orders.pop("_merge").astype(str)
save_interim(items_orders, "items_orders_merged.csv")

# 4.3 items ↔ products: many item rows to one product
items_products = items.merge(products, on="product_id", how="left", validate="m:1", indicator=True)
log_join("items_products", len(items), len(products), len(items_products), "product_id", "product_id", "m:1")
items_products["product_join_status"] = items_products.pop("_merge").astype(str)
save_interim(items_products, "items_products_merged.csv")

# 4.4 orders ↔ customers
orders_customers = orders.merge(customers, on="customer_id", how="left", validate="m:1", suffixes=("", "_customer"), indicator=True)
log_join("orders_customers", len(orders), len(customers), len(orders_customers), "customer_id", "customer_id", "m:1")
orders_customers["customer_join_status"] = orders_customers.pop("_merge").astype(str)
save_interim(orders_customers, "orders_customers_merged.csv")

# 4.5 orders ↔ geography
orders_geography = orders.merge(geography, on="zip", how="left", validate="m:1", suffixes=("", "_geo"), indicator=True)
log_join("orders_geography", len(orders), len(geography), len(orders_geography), "zip", "zip", "m:1")
orders_geography["geo_join_status"] = orders_geography.pop("_merge").astype(str)
save_interim(orders_geography, "orders_geography_merged.csv")

# 4.6 orders ↔ shipments: 1:0/1
orders_shipments = orders.merge(shipments, on="order_id", how="left", validate="1:1", indicator=True)
log_join("orders_shipments", len(orders), len(shipments), len(orders_shipments), "order_id", "order_id", "1:0/1")
orders_shipments["shipment_join_status"] = orders_shipments.pop("_merge").astype(str)
orders_shipments["ship_lead_days"] = (orders_shipments["ship_date"] - orders_shipments["order_date"]).dt.days
orders_shipments["delivery_lead_days"] = (orders_shipments["delivery_date"] - orders_shipments["order_date"]).dt.days
save_interim(orders_shipments, "orders_shipments_final.csv")

pd.DataFrame(join_logs)

Saved orders_payments_merged.csv: (646945, 12)
Saved items_orders_merged.csv: (714669, 15)
Saved items_products_merged.csv: (714669, 15)
Saved orders_customers_merged.csv: (646945, 15)
Saved orders_geography_merged.csv: (646945, 12)
Saved orders_shipments_final.csv: (646945, 14)


,join_name,left_rows,right_rows,output_rows,growth_vs_left,left_key,right_key,expected,status,note
0,orders_payments,646945,646945,646945,1.0000,order_id,order_id,1:1,PASS,
1,items_orders,714669,646945,714669,1.0000,order_id,order_id,m:1,PASS,
2,items_products,714669,2412,714669,1.0000,product_id,product_id,m:1,PASS,
3,orders_customers,646945,121930,646945,1.0000,customer_id,customer_id,m:1,PASS,
4,orders_geography,646945,39948,646945,1.0000,zip,zip,m:1,PASS,
5,orders_shipments,646945,566067,646945,1.0000,order_id,order_id,1:0/1,PASS,


## 5. Build item-level fact table

Grain mục tiêu: mỗi dòng order-item. Sau khi gắn returns/reviews, có thể xuất hiện quan hệ 1:n nếu một dòng sản phẩm có nhiều return/review. Vì vậy bảng fact chính sẽ dùng aggregation của returns/reviews trước khi merge để giữ grain ổn định.

In [ ]:
# Aggregate returns at order-product level before joining
returns_agg = (returns
    .groupby(["order_id", "product_id"], as_index=False)
    .agg(
        return_records=("return_id", "nunique"),
        return_quantity=("return_quantity", "sum"),
        refund_amount=("refund_amount", "sum"),
        first_return_date=("return_date", "min"),
        top_return_reason=("return_reason", lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    )
)

reviews_agg = (reviews
    .groupby(["order_id", "product_id"], as_index=False)
    .agg(
        review_records=("review_id", "nunique"),
        avg_rating=("rating", "mean"),
        first_review_date=("review_date", "min")
    )
)

item_fact = (items
    .merge(orders, on="order_id", how="left", validate="m:1")
    .merge(products, on="product_id", how="left", validate="m:1")
    .merge(customers[["customer_id", "signup_date", "gender", "age_group", "acquisition_channel"]], on="customer_id", how="left", validate="m:1")
    .merge(geography[["zip", "region", "district"]], on="zip", how="left", validate="m:1")
    .merge(payments[["order_id", "payment_value", "installments"]], on="order_id", how="left", validate="m:1")
    .merge(shipments[["order_id", "ship_date", "delivery_date", "shipping_fee"]], on="order_id", how="left", validate="m:1")
    .merge(returns_agg, on=["order_id", "product_id"], how="left")
    .merge(reviews_agg, on=["order_id", "product_id"], how="left")
)

item_fact["line_revenue"] = item_fact["quantity"] * item_fact["unit_price"]
item_fact["line_cogs"] = item_fact["quantity"] * item_fact["cogs"]
item_fact["gross_profit"] = item_fact["line_revenue"] - item_fact["line_cogs"]
item_fact["gross_margin"] = np.where(item_fact["line_revenue"] > 0, item_fact["gross_profit"] / item_fact["line_revenue"], np.nan)
item_fact["has_promo"] = item_fact["promo_id"].notna() | item_fact["promo_id_2"].notna()
item_fact["is_returned"] = item_fact["return_records"].fillna(0).gt(0)
item_fact["order_month"] = item_fact["order_date"].dt.to_period("M").astype(str)

save_interim(item_fact, "1_fact_order_item_enriched.csv")
item_fact.head()

Saved 1_fact_order_item_enriched.csv: (714669, 47)


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,product_name,category,segment,size,color,price,cogs,signup_date,gender,age_group,acquisition_channel,region,district,payment_value,installments,ship_date,delivery_date,shipping_fee,return_records,return_quantity,refund_amount,first_return_date,top_return_reason,review_records,avg_rating,first_review_date,line_revenue,line_cogs,gross_profit,gross_margin,has_promo,is_returned,order_month
0,1,2400,7,"1,138.2200",0.0000,NaN,NaN,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,VietMotion YY-09,GenZ,Trendy,S,red,"1,109.2611","1,053.7980",2020-06-06,Female,35-44,social_media,East,District #02,"7,967.5400",3,2012-07-07,2012-07-11,1.3700,NaN,NaN,NaN,NaT,NaN,1.0000,5.0000,2012-07-24,"7,967.5400","7,376.5861",590.9539,0.0742,False,False,2012-07
1,2,609,7,"10,166.2500",0.0000,NaN,NaN,2012-07-04,58621,1330,returned,cod,mobile,paid_search,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,"10,426.5710","8,987.7042",2021-11-03,Female,18-24,social_media,East,District #02,"71,163.7500",1,2012-07-06,2012-07-10,2.6000,1.0000,6.0000,"52,458.0100",2012-07-25,late_delivery,NaN,NaN,NaT,"71,163.7500","62,913.9296","8,249.8204",0.1159,False,True,2012-07
2,3,396,3,"11,220.3300",0.0000,NaN,NaN,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,SaigonFlex UM-01,Streetwear,Balanced,S,green,"11,028.4287","10,091.0123",2020-09-18,Female,35-44,direct,East,District #02,"33,660.9900",3,2012-07-04,2012-07-07,2.3800,NaN,NaN,NaN,NaT,NaN,1.0000,5.0000,2012-08-03,"33,660.9900","30,273.0368","3,387.9532",0.1006,False,False,2012-07
3,4,635,5,"10,639.2500",0.0000,NaN,NaN,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,SaigonFlex UC-00,Streetwear,Everyday,XL,purple,"10,745.2206","9,205.4305",2016-05-29,Male,45-54,direct,East,District #02,"53,196.2500",3,2012-07-05,2012-07-11,2.4900,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaT,"53,196.2500","46,027.1524","7,169.0976",0.1348,False,False,2012-07
4,6,1935,1,"1,597.8400",0.0000,NaN,NaN,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,UrbanVN RP-10,Outdoor,Activewear,XL,purple,"1,609.9115","1,048.6964",2017-07-11,Male,18-24,social_media,East,District #02,"1,597.8400",1,2012-07-09,2012-07-16,25.7900,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaT,"1,597.8400","1,048.6964",549.1436,0.3437,False,False,2012-07


## 6. Build order-level fact table

In [ ]:
order_item_agg = (item_fact
    .groupby("order_id", as_index=False)
    .agg(
        order_items=("product_id", "count"),
        order_units=("quantity", "sum"),
        order_revenue_from_items=("line_revenue", "sum"),
        order_cogs_from_items=("line_cogs", "sum"),
        order_discount=("discount_amount", "sum"),
        any_promo=("has_promo", "max"),
        any_return=("is_returned", "max"),
        avg_order_rating=("avg_rating", "mean")
    )
)

order_fact = (orders
    .merge(order_item_agg, on="order_id", how="left", validate="1:1")
    .merge(payments[["order_id", "payment_value", "installments"]], on="order_id", how="left", validate="1:1")
    .merge(shipments[["order_id", "ship_date", "delivery_date", "shipping_fee"]], on="order_id", how="left", validate="1:1")
    .merge(customers[["customer_id", "signup_date", "gender", "age_group", "acquisition_channel"]], on="customer_id", how="left", validate="m:1")
    .merge(geography[["zip", "region", "district"]], on="zip", how="left", validate="m:1")
)
order_fact["ship_lead_days"] = (order_fact["ship_date"] - order_fact["order_date"]).dt.days
order_fact["delivery_lead_days"] = (order_fact["delivery_date"] - order_fact["order_date"]).dt.days
order_fact["order_month"] = order_fact["order_date"].dt.to_period("M").astype(str)

save_interim(order_fact, "2_fact_order_enriched.csv")
order_fact.head()

Saved 2_fact_order_enriched.csv: (646945, 30)


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,order_items,order_units,order_revenue_from_items,order_cogs_from_items,order_discount,any_promo,any_return,avg_order_rating,payment_value,installments,ship_date,delivery_date,shipping_fee,signup_date,gender,age_group,acquisition_channel,region,district,ship_lead_days,delivery_lead_days,order_month
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,1,7,"7,967.5400","7,376.5861",0.0000,False,False,5.0000,"7,967.5400",3,2012-07-07,2012-07-11,1.3700,2020-06-06,Female,35-44,social_media,East,District #02,3.0000,7.0000,2012-07
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search,1,7,"71,163.7500","62,913.9296",0.0000,False,True,NaN,"71,163.7500",1,2012-07-06,2012-07-10,2.6000,2021-11-03,Female,18-24,social_media,East,District #02,2.0000,6.0000,2012-07
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,1,3,"33,660.9900","30,273.0368",0.0000,False,False,5.0000,"33,660.9900",3,2012-07-04,2012-07-07,2.3800,2020-09-18,Female,35-44,direct,East,District #02,0.0000,3.0000,2012-07
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,1,5,"53,196.2500","46,027.1524",0.0000,False,False,NaN,"53,196.2500",3,2012-07-05,2012-07-11,2.4900,2016-05-29,Male,45-54,direct,East,District #02,1.0000,7.0000,2012-07
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,1,1,"1,597.8400","1,048.6964",0.0000,False,False,NaN,"1,597.8400",1,2012-07-09,2012-07-16,25.7900,2017-07-11,Male,18-24,social_media,East,District #02,3.0000,10.0000,2012-07


## 7. Build marts for EDA and modeling

In [ ]:
# Customer 360
customer_360 = (customers
    .merge(order_fact.groupby("customer_id", as_index=False).agg(
        orders=("order_id", "nunique"),
        total_revenue=("order_revenue_from_items", "sum"),
        total_units=("order_units", "sum"),
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max"),
        return_orders=("any_return", "sum"),
        promo_orders=("any_promo", "sum")
    ), on="customer_id", how="left")
)
customer_360["orders"] = customer_360["orders"].fillna(0).astype(int)
customer_360["return_order_rate"] = np.where(customer_360["orders"] > 0, customer_360["return_orders"] / customer_360["orders"], np.nan)
customer_360["promo_order_rate"] = np.where(customer_360["orders"] > 0, customer_360["promo_orders"] / customer_360["orders"], np.nan)
save_mart(customer_360, "3_customer_360.csv")

# Product 360
product_360 = (products
    .merge(item_fact.groupby("product_id", as_index=False).agg(
        item_lines=("order_id", "count"),
        units_sold=("quantity", "sum"),
        revenue=("line_revenue", "sum"),
        cogs_total=("line_cogs", "sum"),
        gross_profit=("gross_profit", "sum"),
        promo_lines=("has_promo", "sum"),
        returned_lines=("is_returned", "sum"),
        avg_rating=("avg_rating", "mean")
    ), on="product_id", how="left")
)
for c in ["item_lines", "units_sold", "revenue", "cogs_total", "gross_profit", "promo_lines", "returned_lines"]:
    product_360[c] = product_360[c].fillna(0)
product_360["gross_margin"] = np.where(product_360["revenue"] > 0, product_360["gross_profit"] / product_360["revenue"], np.nan)
product_360["promo_line_rate"] = np.where(product_360["item_lines"] > 0, product_360["promo_lines"] / product_360["item_lines"], np.nan)
product_360["return_line_rate"] = np.where(product_360["item_lines"] > 0, product_360["returned_lines"] / product_360["item_lines"], np.nan)
product_360 = product_360.sort_values("revenue", ascending=False)
product_360["revenue_rank"] = np.arange(1, len(product_360) + 1)
product_360["cum_revenue_share"] = product_360["revenue"].cumsum() / product_360["revenue"].sum()
product_360["abc_class"] = pd.cut(product_360["cum_revenue_share"], bins=[0, 0.80, 0.95, 1.01], labels=["A", "B", "C"], include_lowest=True)
save_mart(product_360, "4_product_360.csv")

# Daily business panel
traffic_daily = traffic.groupby("date", as_index=False).agg(
    sessions=("sessions", "sum"),
    unique_visitors=("unique_visitors", "sum"),
    page_views=("page_views", "sum"),
    avg_bounce_rate=("bounce_rate", "mean"),
    avg_session_duration_sec=("avg_session_duration_sec", "mean")
).rename(columns={"date": "Date"})

orders_daily = order_fact.groupby("order_date", as_index=False).agg(
    orders=("order_id", "nunique"),
    order_units=("order_units", "sum"),
    promo_orders=("any_promo", "sum"),
    return_orders=("any_return", "sum"),
    avg_delivery_lead_days=("delivery_lead_days", "mean")
).rename(columns={"order_date": "Date"})

daily_panel = (sales
    .merge(traffic_daily, on="Date", how="left")
    .merge(orders_daily, on="Date", how="left")
)
daily_panel["gross_profit"] = daily_panel["Revenue"] - daily_panel["COGS"]
daily_panel["gross_margin"] = np.where(daily_panel["Revenue"] > 0, daily_panel["gross_profit"] / daily_panel["Revenue"], np.nan)
daily_panel["year"] = daily_panel["Date"].dt.year
daily_panel["month"] = daily_panel["Date"].dt.month
daily_panel["dayofweek"] = daily_panel["Date"].dt.dayofweek
daily_panel["is_weekend"] = daily_panel["dayofweek"].isin([5, 6]).astype(int)
save_mart(daily_panel, "5_daily_business_panel.csv")

# Product-month panel
item_pm = (item_fact.assign(snapshot_date=item_fact["order_date"].dt.to_period("M").dt.to_timestamp("M"))
    .groupby(["snapshot_date", "product_id"], as_index=False)
    .agg(item_units_sold=("quantity", "sum"), item_revenue=("line_revenue", "sum"), item_gross_profit=("gross_profit", "sum"), promo_lines=("has_promo", "sum"), returned_lines=("is_returned", "sum"))
)
product_month = inventory.merge(products[["product_id", "price", "cogs", "size", "color"]], on="product_id", how="left", suffixes=("", "_master"))
product_month = product_month.merge(item_pm, on=["snapshot_date", "product_id"], how="left")
for c in ["item_units_sold", "item_revenue", "item_gross_profit", "promo_lines", "returned_lines"]:
    product_month[c] = product_month[c].fillna(0)
product_month["inventory_risk_score"] = product_month["stockout_flag"].fillna(0) + product_month["reorder_flag"].fillna(0) - product_month["overstock_flag"].fillna(0)
save_mart(product_month, "6_product_month_panel.csv")

pd.DataFrame({
    "mart": ["customer_360", "product_360", "daily_business_panel", "product_month_panel"],
    "rows": [len(customer_360), len(product_360), len(daily_panel), len(product_month)],
    "cols": [customer_360.shape[1], product_360.shape[1], daily_panel.shape[1], product_month.shape[1]]
})

Saved 3_customer_360.csv: (121930, 16)
Saved 4_product_360.csv: (2412, 22)
Saved 5_daily_business_panel.csv: (3833, 19)
Saved 6_product_month_panel.csv: (60247, 27)


,mart,rows,cols
0,customer_360,121930,16
1,product_360,2412,22
2,daily_business_panel,3833,19
3,product_month_panel,60247,27


## 8. Final validation of generated datamarts

In [ ]:
outputs = {
    "1_fact_order_item_enriched.csv": (INTERIM_DIR, ["order_id", "product_id"], False),
    "2_fact_order_enriched.csv": (INTERIM_DIR, ["order_id"], True),
    "3_customer_360.csv": (MARTS_DIR, ["customer_id"], True),
    "4_product_360.csv": (MARTS_DIR, ["product_id"], True),
    "5_daily_business_panel.csv": (MARTS_DIR, ["Date"], True),
    "6_product_month_panel.csv": (MARTS_DIR, ["snapshot_date", "product_id"], True),
}

output_checks = []
for fname, (folder, key, expect_unique) in outputs.items():
    p = folder / fname
    df = pd.read_csv(p, low_memory=False)
    dup = int(df.duplicated(key).sum()) if set(key).issubset(df.columns) else None
    output_checks.append({
        "file": str(p.relative_to(PROJECT_ROOT)) if str(p).startswith(str(PROJECT_ROOT)) else str(p),
        "rows": len(df),
        "cols": df.shape[1],
        "key": ", ".join(key),
        "duplicate_key_rows": dup,
        "expected_unique": expect_unique,
        "status": "PASS" if (not expect_unique or dup == 0) else "CHECK"
    })

output_checks = pd.DataFrame(output_checks)
save_table(output_checks, "02_output_datamart_checks.csv")
save_table(pd.DataFrame(join_logs), "02_join_logs.csv")
output_checks

Saved table: C:\datathon\reports\tables\02_output_datamart_checks.csv
Saved table: C:\datathon\reports\tables\02_join_logs.csv


,file,rows,cols,key,duplicate_key_rows,expected_unique,status
0,data\interim\1_fact_order_item_enriched.csv,714669,47,"order_id, product_id",16,False,PASS
1,data\interim\2_fact_order_enriched.csv,646945,30,order_id,0,True,PASS
2,data\marts\3_customer_360.csv,121930,16,customer_id,0,True,PASS
3,data\marts\4_product_360.csv,2412,22,product_id,0,True,PASS
4,data\marts\5_daily_business_panel.csv,3833,19,Date,0,True,PASS
5,data\marts\6_product_month_panel.csv,60247,27,"snapshot_date, product_id",0,True,PASS


## 9. Notebook conclusion

Sau notebook này, dùng các bảng trong `data/interim/` và `data/marts/` cho EDA. Nếu một bảng item-level bị `CHECK` do grain `(order_id, product_id)` không unique, không nhất thiết phải sửa nếu phân tích đang dùng aggregation đúng trước khi join 1:n.